## Εργασία 3 ##

Καλωσήρθατε στην τρίτη σας εργασία. Η εργασία αυτή έχει σκοπό να σας βοηθήσει να εμπεδώσετε τα σύνολα μοντέλων.

Στην εργασία αυτή θα πρέπει να συμπληρώσετε κώδικα Python 3 στα σημεία που αναφέρουν # YOUR CODE HERE. Μην τροποποιείτε τον κώδικα που βρίσκεται εκτός αυτών των περιοχών.

Πρωτού παραδόσετε την εργασία σας σιγουρευτείτε ότι ο κώδικας σε όλα τα κελιά τρέχει σωστά. Για το σκοπό αυτό επιλέξτε από το μενού **Χρόνος εκτέλεσης (runtime) -> Επανεκίνηση περιόδου λειτουργίας και εκτέλεση όλων**.

Συμπληρώστε το όνομα (**NAME**) και το **AEM** σας παρακάτω **εντός των διπλών εισαγωγικών**:

In [1]:
NAME = "Lavdakis Giorgos"
AEM = "10672"

**1 (2 μονάδες)** Σε αυτήν την εργασία θα δουλέψουμε με ένα σύνολο δεδομένων δυαδικής ταξινόμησης, από το δημόσιο αποθετήριο δεδομένων και πειραμάτων μηχανικής μάθησης [openml.org](https://www.openml.org/). Μπορείτε να δείτε πως φορτώνουμε σύνολα δεδομένων από το openml.org μέσα από την sklearn μελετώντας την αντίστοιχη [τεκμηρίωση](https://scikit-learn.org/stable/datasets/loading_other_datasets.html#downloading-datasets-from-the-openml-org-repository). Συγκεκριμένα θέλουμε να δουλέψουμε με την έκδοση 1 του Pima Indians Diabetes Database, το οποίο αφορά τη διάγνωση διαβήτη. Χωρίστε το σύνολο δεδομένων σε δεδομένα εκπαίδευσης (X_train, y_train) και ελέγχου (X_test, y_test) σε ποσοστό 70% και 30% αντίστοιχα με τη συνάρτηση train_test_split (τιμή για random_state βάλτε 0).

In [2]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

diabetes = fetch_openml(name='Diabetes', version=1) # Load the diabetes dataset version 1 from the openml.org repository
X = diabetes.data 
y = diabetes.target
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0, test_size=0.3, shuffle=True) 

In [3]:
"""Τεστ ορθής ανάγνωσης και διαχωρισμού του συνόλου δεδομένων"""
assert X_train['mass'][0] == 33.6
assert X_test['pedi'][34] == 0.512

**2 (4 μονάδες)** Υλοποιήστε μια ντετερμινιστική εκδοχή της μεθόδου των τυχαίων υποχώρων, η οποία χτίζει τόσα μοντέλα όσες και οι μεταβλητές εισόδου, κάθε ένα από τα οποία αγνοεί και μία διαφορετική μεταβλητή εισόδου. Π.χ. το πρώτο μοντέλο αγνοεί την πρώτη, το δεύτερο αγνοεί τη δεύτερη κτλ. Χρησιμοποιήστε τη συνάρτηση clone από το sklearn.base για να δημιουργήστε αντίγραφο του βασικού μοντέλου σε κάθε επανάληψη.

In [4]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone

class RandomSubspaceDet:
    def __init__(self, estimator=DecisionTreeClassifier()):
      self.estimator = estimator # Base model - will be cloned for each sub-model
      self.models = [] # List to store the trained models
      self.ignored_features = [] # Stores the index of the ignored feature in each model

    def fit(self, X_train, y_train):
      self.models = [] # Clear any previous models
      self.ignored_features = []
      n_features = X_train.shape[1] # Get the number of input features

      for i in range(n_features): # Loop over each feature
        features_to_use = [j for j in range(n_features) if j != i] # Create a list of all features except the one at index i

        X_subset = X_train.iloc[:, features_to_use] # Select the desired features 

        model = clone(self.estimator) 
        model.fit(X_subset, y_train) 
        self.models.append(model) # Store the model 
        self.ignored_features.append(i) # Store which feature was ignored

    def predict(self, X):
      predictions = [] # Empty list to collect predictions from each model

      for i, model in enumerate(self.models): # Loop over each model
        features_to_use = [j for j in range(X.shape[1]) if j != self.ignored_features[i]] # Select the same subset of features used during training - excluding the ignored one

        X_subset = X.iloc[:, features_to_use] # Select the desired features 

        pred = model.predict(X_subset) 
        predictions.append(pred) 

      predictions = np.array(predictions) # Convert predictions to array with shape (n_models, n_samples) - each row is predictions from each model

      n_samples = predictions.shape[1] # Total number of examples to predict
      final_preds = [] # List that will collect the majority prediction for each sample

      for i in range(n_samples): # Loop over each sample
        labels, counts = np.unique(predictions[:, i], return_counts=True) # Collect predictions from all models for sample i
        # labels contain the feature names and counts how many times each label was predicted
        majority_label = labels[np.argmax(counts)] # Return the index of the most frequent label
        final_preds.append(majority_label) # Append the most common prediction for this sample to the final result

      return np.array(final_preds) # Return the final prediction for each test sample

In [5]:
"""Τεστ ορθής υλοποίησης RandomSubspaceDet"""
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

rs = RandomSubspaceDet(estimator=DecisionTreeClassifier(random_state=1))
rs.fit(X_train, y_train)
assert round(accuracy_score(rs.predict(X_test), y_test), 4) == 0.7446

**3 (4 μονάδες)** Υλοποιήστε τη μέθοδο AdaBoost όπως παρουσιάστηκε στο μάθημα. Χρησιμοποιήστε τη συνάρτηση clone από το sklearn.base για να δημιουργήστε αντίγραφο του βασικού μοντέλου σε κάθε επανάληψη. Χρησιμοποιήστε την παράμετρο sample_weight της fit του βασικού μοντέλου για να ορίσετε τα βάρη των παραδειγμάτων εκπαίδευσης.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone

class AdaBoost:
    def __init__(self, n_estimators=20, estimator=DecisionTreeClassifier(max_depth=1)):
      self.n_estimators = n_estimators # Number of weak learners N
      self.estimator = estimator # Base model - will be cloned for each sub-model

    def fit(self, X_train, y_train):
      n_samples = X_train.shape[0] # Number of training samples m

      self.classes = np.unique(y_train) 
      y = np.array(y_train) 

      weights = np.ones(n_samples) / n_samples 

      self.estimators = [] # List to store models
      self.errors = [] # List to store errors of each model

      for t in range(self.n_estimators): 
        estimator = clone(self.estimator) 
        estimator.fit(X_train, y, sample_weight=weights) 
        predictions = np.array(estimator.predict(X_train)) 

        incorrect = (predictions != y) # Number of wrong predictions
        error = np.sum(weights * incorrect) # Weighted error - sum of the weights of the wrong predictions
        self.errors.append(error) 

        if error >= 0.5 or error == 0: # Break if error is too high or takes the minimum value
          break

        self.estimators.append(estimator) 

        for i in range(n_samples): 
          if y[i] == predictions[i]: # If correctly predicted
            weights[i] = weights[i] * error / (1 - error) # Update weights

        for i in range(self.n_estimators): 
          weights = weights / np.sum(weights) # Normalize weights

      return self

    def predict(self, X):
      n_samples = X.shape[0] 
      n_classes = len(self.classes) # Number of classes
      scores = np.zeros((n_samples, n_classes)) # Initialize scores for each class for each sample

      for t, (estimator, error) in enumerate(zip(self.estimators, self.errors)): 
        preds = np.array(estimator.predict(X)) 

        for i in range(n_samples): 
          class_idx = np.where(self.classes == preds[i])[0][0] # Find index of predicted class
          scores[i, class_idx] += np.log((1 - error) / error) # Add weighted vote for the predicted class for each sample

      return self.classes[np.argmax(scores, axis=1)] # Return class with highest score

In [7]:
"""Τεστ ορθής υλοποίησης AdaBoost"""
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

ab = AdaBoost(n_estimators=20, estimator=DecisionTreeClassifier(max_depth=2, random_state=1))
ab.fit(X_train, y_train)
assert round(accuracy_score(ab.predict(X_test), y_test), 4) == 0.7706


In [8]:
#Ίδιο αποτέλεσμα παίρνουμε και με την κλάση της sklearn. Ωστόσο υπάρχουν γενικότερα διαφορές στην υλοποίηση της σε σ΄χέση με αυτήν που είδαμε στη θεωρία, όπως η υποστήριξη παραπάνω από 2 κλάσεων και ο διαφορετικός χειρισμός της τυχαιότητας στην εκπαίδευση των επί μέρους μοντέλων.
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

ab = AdaBoostClassifier(n_estimators=20, estimator=DecisionTreeClassifier(max_depth=2, random_state=1))
ab.fit(X_train, y_train)
assert round(accuracy_score(ab.predict(X_test), y_test), 4) == 0.7706